# 20 Preguntas de Negocio

In [1]:
# PostgreSQL connection
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os

load_dotenv()

def connect_to_db():
    try:
        connection = psycopg2.connect(
            dbname=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT")
        )
        print("Connection to PostgreSQL database successful")
        return connection
    except Exception as e:
        print(f"Error connecting to PostgreSQL database: {e}")
        return None
    
days_list = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

months_list = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

### 1. Viajes por mes en 2024:

In [49]:
query_1 = """
SELECT d.month, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.month
ORDER BY d.month;
"""

connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_1)
        results = cursor.fetchall()
        print("Number of rows per month of 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Number of rows per month of 2024 in the analytics_gold.fct_trips table:
Month: January, Trip Count: 2985411
Month: February, Trip Count: 3024821
Month: March, Trip Count: 3595427
Month: April, Trip Count: 3526712
Month: May, Trip Count: 3735534
Month: June, Trip Count: 3544893
Month: July, Trip Count: 3078104
Month: August, Trip Count: 2977751
Month: September, Trip Count: 3631073
Month: October, Trip Count: 3828252
Month: November, Trip Count: 3636752
Month: December, Trip Count: 3651549


Se puede observar que los meses más ajetreados del servicio de taxis de NYC son Mayo, Septiembre y Octubre, mientras que los más tranquilos son Enero, Agosto y Febrero de 2024.

### 2. Viajes por *service_type* y mes

In [52]:
query_2 = """
SELECT d.month, s.service_name, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_service_type s ON f.service_type_key = s.service_type_key
GROUP BY d.month, s.service_name
ORDER BY s.service_name, d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_2)
        results = cursor.fetchall()
        print("Number of trips per year, month, and service type in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Service Type: {row[1]}, Month: {months_list[int(row[0] - 1)]}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Number of trips per year, month, and service type in the analytics_gold.fct_trips table:
Service Type: green, Month: January, Trip Count: 469606
Service Type: green, Month: February, Trip Count: 466788
Service Type: green, Month: March, Trip Count: 516900
Service Type: green, Month: April, Trip Count: 498122
Service Type: green, Month: May, Trip Count: 522846
Service Type: green, Month: June, Trip Count: 484794
Service Type: green, Month: July, Trip Count: 449764
Service Type: green, Month: August, Trip Count: 447832
Service Type: green, Month: September, Trip Count: 474094
Service Type: green, Month: October, Trip Count: 480762
Service Type: green, Month: November, Trip Count: 449500
Service Type: green, Month: December, Trip Count: 380058
Service Type: yellow, Month: January, Trip Count: 23664386
Service Type: yellow, Month: February, Trip Count: 24689156
Service Type: yellow, Month: March, Trip Count: 29188160
Service Type: yellow, Month:

Para el servicio de taxis verdes de NYC se tiene que los meses más activos son Mayo, Marzo y Junio, mientras que para los amarillos son Mayo, Octubre y Marzo, mostrando una tendencia general de aumento en el tráfico poblacional y no específicamente para uno de los dos servicios. 

### 3. Top 10 zonas de pickup (total 2024)

In [9]:
query_3 = """
SELECT z.zone_name, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY z.zone_name
ORDER BY total_viajes DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_3)
        results = cursor.fetchall()
        print("Top 10 pickup zones with the most trips in 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones with the most trips in 2024 in the analytics_gold.fct_trips table:
Zone Name: JFK Airport, Trip Count: 1910212
Zone Name: Upper East Side South, Trip Count: 1892766
Zone Name: Midtown Center, Trip Count: 1886869
Zone Name: Upper East Side North, Trip Count: 1718364
Zone Name: Midtown East, Trip Count: 1399886
Zone Name: Times Sq/Theatre District, Trip Count: 1362163
Zone Name: Penn Station/Madison Sq West, Trip Count: 1340568
Zone Name: Lincoln Square East, Trip Count: 1299317
Zone Name: LaGuardia Airport, Trip Count: 1274993
Zone Name: Murray Hill, Trip Count: 1160460


Siendo NYC una ciudad muy turística, es de esperar que la mayoría de tráfico de los taxis provenga de gente llegando al aeropuerto internacional JFK en 2024. Sin embargo, es inesperado observar que LaGuardia no se encuentre más arriba en el top dado que también es un aeropuerto, aunque se puede entender dado que no es internacional al nivel del JFK.

### 4. Top 10 zonas de dropoff (total 2024)

In [10]:
query_4 = """
SELECT z.zone_name, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.do_zone_key = z.zone_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY z.zone_name
ORDER BY total_viajes DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_4)
        results = cursor.fetchall()
        print("Top 10 dropoff zones with the most trips in 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 dropoff zones with the most trips in 2024 in the analytics_gold.fct_trips table:
Zone Name: Upper East Side North, Trip Count: 1807540
Zone Name: Upper East Side South, Trip Count: 1714779
Zone Name: Midtown Center, Trip Count: 1521463
Zone Name: Times Sq/Theatre District, Trip Count: 1286396
Zone Name: Murray Hill, Trip Count: 1199473
Zone Name: Midtown East, Trip Count: 1171121
Zone Name: Lincoln Square East, Trip Count: 1137175
Zone Name: Upper West Side South, Trip Count: 1135542
Zone Name: East Chelsea, Trip Count: 1063496
Zone Name: Lenox Hill West, Trip Count: 1058176


Siento los vecindarios del Upper East Side de NYC los más conocidos y solicitados por turistas, era de esperarse que sean los puntos donde los taxis dejan a un mayor número de pasajeros. Esto demuestra una vez más que NYC es una ciudad de naturaleza turística.

### 5. Top 5 *boroughs* por mes (pickup)

In [19]:
query_5 = """
WITH borough_counts AS (
    SELECT d.month, z.borough, COUNT(*) AS total_viajes,
           ROW_NUMBER() OVER(PARTITION BY d.month ORDER BY COUNT(*) DESC) as rank
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
    JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
    GROUP BY d.month, z.borough
)
SELECT month, borough, total_viajes
FROM borough_counts
WHERE rank <= 5
ORDER BY month, total_viajes DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_5)
        results = cursor.fetchall()
        print("Top 5 boroughs with the most trips per month in 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Borough: {row[1]}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 5 boroughs with the most trips per month in 2024 in the analytics_gold.fct_trips table:
Month: January, Borough: Manhattan, Trip Count: 10707270
Month: January, Borough: Queens, Trip Count: 1075823
Month: January, Borough: Brooklyn, Trip Count: 155928
Month: January, Borough: Unknown, Trip Count: 81075
Month: January, Borough: Bronx, Trip Count: 34724
Month: February, Borough: Manhattan, Trip Count: 11221813
Month: February, Borough: Queens, Trip Count: 1026254
Month: February, Borough: Brooklyn, Trip Count: 195731
Month: February, Borough: Unknown, Trip Count: 78834
Month: February, Borough: Bronx, Trip Count: 41743
Month: March, Borough: Manhattan, Trip Count: 13073387
Month: March, Borough: Queens, Trip Count: 1343960
Month: March, Borough: Brooklyn, Trip Count: 269615
Month: March, Borough: Unknown, Trip Count: 86991
Month: March, Borough: Bronx, Trip Count: 61014
Month: April, Borough: Manhattan, Trip Count: 12708210
Month: April, B

Se puede observar como todos los meses, sin excepción, el orden de los municipios de NYC donde se recogen pasajeros se mantiene constante, siendo el más transitado Manhattan y el menos transitado el Bronx, pero aún así con un número significativo de viajes registrados.

### 6. Horas pico (top 5) por cada día de la semana

In [20]:
query_6 = """
WITH hourly_counts AS (
    SELECT d.day_of_week, EXTRACT(HOUR FROM f.pickup_ts) AS hora, COUNT(*) AS total_viajes,
           ROW_NUMBER() OVER(PARTITION BY d.day_of_week ORDER BY COUNT(*) DESC) as rank
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
    GROUP BY d.day_of_week, hora
)
SELECT day_of_week, hora, total_viajes
FROM hourly_counts
WHERE rank <= 5
ORDER BY day_of_week, total_viajes DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_6)
        results = cursor.fetchall()
        print("Top 5 hours with the most trips per day of the week in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Day of Week: {days_list[int(row[0]) - 1]}, Hour: {int(row[1])}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 5 hours with the most trips per day of the week in the analytics_gold.fct_trips table:
Day of Week: Monday, Hour: 18, Trip Count: 1502911
Day of Week: Monday, Hour: 17, Trip Count: 1474289
Day of Week: Monday, Hour: 15, Trip Count: 1349761
Day of Week: Monday, Hour: 16, Trip Count: 1340784
Day of Week: Monday, Hour: 14, Trip Count: 1291775
Day of Week: Tuesday, Hour: 18, Trip Count: 1778223
Day of Week: Tuesday, Hour: 17, Trip Count: 1667511
Day of Week: Tuesday, Hour: 19, Trip Count: 1506045
Day of Week: Tuesday, Hour: 21, Trip Count: 1471169
Day of Week: Tuesday, Hour: 16, Trip Count: 1469483
Day of Week: Wednesday, Hour: 18, Trip Count: 1852493
Day of Week: Wednesday, Hour: 17, Trip Count: 1731430
Day of Week: Wednesday, Hour: 19, Trip Count: 1629727
Day of Week: Wednesday, Hour: 21, Trip Count: 1557395
Day of Week: Wednesday, Hour: 16, Trip Count: 1514376
Day of Week: Thursday, Hour: 18, Trip Count: 1899799
Day of Week: Thursday, Hou

Siendo NYC una ciudad con mucho movimiento capital, la capital financiera del mundo de hecho, con un gran número de empleos de naturaleza 9 a 5, es de esperar que a las 6 pm se registre el mayor número de pasajeros recogidos, ya que en su mayoría están finalizando sus jornadas laborales. Así mismo, los domingos es más común trabajar hasta medio día o salir a la hora del almuerzo.

### 7. Distribución de viajes por día de la semana (ranking)

In [21]:
query_7 = """
SELECT d.day_of_week, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.day_of_week
ORDER BY total_viajes DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_7)
        results = cursor.fetchall()
        print("Total trips per day of the week in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Day of Week: {days_list[int(row[0]) - 1]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total trips per day of the week in the analytics_gold.fct_trips table:
Day of Week: Thursday, Trip Count: 25567049
Day of Week: Wednesday, Trip Count: 24674116
Day of Week: Friday, Trip Count: 24620965
Day of Week: Saturday, Trip Count: 24597764
Day of Week: Tuesday, Trip Count: 23422015
Day of Week: Sunday, Trip Count: 20873448
Day of Week: Monday, Trip Count: 20429426


Observando que los días más ajetreados de la semana son los Jueves y los Miércoles, una investigación sociológica revela que esos días son los que las personas empleadas en horarios laborales comunes (9 a 5) falta en menor cantidad al trabajo por el deseo de dejar la menor cantidad de actividades pendientes durante el fin de semana. Con lo que suelen trabajar horas extra en promedio aquellos días.

### 8. Ingreso total por mes

In [25]:
query_8 = """
SELECT d.month, SUM(total_amount) AS ingreso_total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_8)
        results = cursor.fetchall()
        print("Total income per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Total Income: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total income per month in the analytics_gold.fct_trips table:
Month: January, Total Income: 306813207.14
Month: February, Total Income: 314218156.24
Month: March, Total Income: 385060198.68
Month: April, Total Income: 381312241.88
Month: May, Total Income: 421143328.69
Month: June, Total Income: 402521154.70
Month: July, Total Income: 352388085.17
Month: August, Total Income: 338092739.22
Month: September, Total Income: 389677515.39
Month: October, Total Income: 424087141.96
Month: November, Total Income: 385454309.34
Month: December, Total Income: 291732404.54


Esta inforación de los ingresos mensuales es consistente con el número de viajes registrados por cada mes de trabajo de los taxis amarillos de NYC debido a la inmensa diferencia en la cantidad de viajes respecto a su contraparte verde. Así mismo, estos datos también son consistentes con la información de los taxis verdes, ya que en ambos casos, los meses más ajetreados se solapan entre sí.

### 9. Ingreso total por *service_type* y mes

In [53]:
query_9 = """
SELECT d.month, s.service_name, SUM(total_amount) AS ingreso_total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_service_type s ON f.service_type_key = s.service_type_key
GROUP BY s.service_name, d.month
ORDER BY s.service_name, d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_9)
        results = cursor.fetchall()
        print("Total income per month and service type in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Service Type: {row[1]}, Month: {months_list[int(row[0]) - 1]}, Total Income: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total income per month and service type in the analytics_gold.fct_trips table:
Service Type: green, Month: January, Total Income: 9891933.72
Service Type: green, Month: February, Total Income: 9867924.18
Service Type: green, Month: March, Total Income: 11190832.40
Service Type: green, Month: April, Total Income: 11083985.38
Service Type: green, Month: May, Total Income: 12132929.40
Service Type: green, Month: June, Total Income: 11408905.22
Service Type: green, Month: July, Total Income: 10562662.56
Service Type: green, Month: August, Total Income: 10967200.80
Service Type: green, Month: September, Total Income: 11965021.72
Service Type: green, Month: October, Total Income: 11384474.00
Service Type: green, Month: November, Total Income: 10355869.30
Service Type: green, Month: December, Total Income: 8767306.70
Service Type: yellow, Month: January, Total Income: 603734480.56
Service Type: yellow, Month: February, Total Income: 618568388.30
Se

Los taxis verdes registran el mayor número de ingresos en los meses de Mayo, Septiembre y Junio, mientras que para los taxis amarillos se tienen los meses de Mayo, Octubre y Septiembre.

### 10. tip% promedio por mes

In [28]:
query_10 = """
SELECT d.month, AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_10)
        results = cursor.fetchall()
        print("Average tip percentage per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Average Tip Percentage: {row[1]:.2f}%")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average tip percentage per month in the analytics_gold.fct_trips table:
Month: January, Average Tip Percentage: 21.50%
Month: February, Average Tip Percentage: 20.46%
Month: March, Average Tip Percentage: 20.15%
Month: April, Average Tip Percentage: 20.95%
Month: May, Average Tip Percentage: 20.11%
Month: June, Average Tip Percentage: 20.48%
Month: July, Average Tip Percentage: 20.01%
Month: August, Average Tip Percentage: 19.90%
Month: September, Average Tip Percentage: 19.49%
Month: October, Average Tip Percentage: 20.45%
Month: November, Average Tip Percentage: 20.31%
Month: December, Average Tip Percentage: 21.38%


Se puede ver que los porcentajes de propina aumentan en los meses de Diciembre y Enero, y lo hacen de forma bastante notable respecto al resto de meses. Esto podría indicar que durante los meses de feriados más importantes como Navidad y Año Nuevo, el contexto de dichas celebraciones parece apelar a la solidaridad de las personas.

### 11. tip% por *borough* y mes

In [55]:
query_11 = """
SELECT d.month, z.borough, AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE z.borough IS NOT NULL
GROUP BY z.borough, d.month
ORDER BY d.month, avg_tip_pct DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_11)
        results = cursor.fetchall()
        print("Average tip percentage per month and borough in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Borough: {row[1]}, Average Tip Percentage: {row[2]:.2f}%")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average tip percentage per month and borough in the analytics_gold.fct_trips table:
Month: January, Borough: EWR, Average Tip Percentage: 47.17%
Month: January, Borough: Manhattan, Average Tip Percentage: 21.21%
Month: January, Borough: Unknown, Average Tip Percentage: 20.16%
Month: January, Borough: Queens, Average Tip Percentage: 19.40%
Month: January, Borough: Staten Island, Average Tip Percentage: 15.42%
Month: January, Borough: Brooklyn, Average Tip Percentage: 9.12%
Month: January, Borough: Bronx, Average Tip Percentage: 4.07%
Month: February, Borough: EWR, Average Tip Percentage: 415.34%
Month: February, Borough: Unknown, Average Tip Percentage: 21.04%
Month: February, Borough: Manhattan, Average Tip Percentage: 20.92%
Month: February, Borough: Queens, Average Tip Percentage: 16.02%
Month: February, Borough: Brooklyn, Average Tip Percentage: 9.29%
Month: February, Borough: Staten Island, Average Tip Percentage: 7.72%
Month: February, 

En todos los meses, el municipio que mayor porcentaje de propina registró fue EWR, siendo este del Aeropuerto Internacional Newark Liberty, siendo esto probablemente porque el aeropuerto se encuentra casi a las afueras de New York, y representa un largo y atareado viaje, con lo que los pasajeros se podrían encontrar más agradecidos por ello. Ahora bien, también es una posibilidad que los taxis soliciten un costo mayor al que registran y la diferencia se considere como propina debido a la razón anterior.

### 12. Top 10 zonas por ingreso total (pickup)

In [31]:
query_12 = """
SELECT z.zone_name, SUM(total_amount) AS ingreso_total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY z.zone_name
ORDER BY ingreso_total DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_12)
        results = cursor.fetchall()
        print("Top 10 pickup zones with the highest total income in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Total Income: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones with the highest total income in the analytics_gold.fct_trips table:
Zone Name: JFK Airport, Total Income: 575931894.53
Zone Name: LaGuardia Airport, Total Income: 304471456.37
Zone Name: Midtown Center, Total Income: 169643637.96
Zone Name: Upper East Side South, Total Income: 145404022.56
Zone Name: Times Sq/Theatre District, Total Income: 140235336.76
Zone Name: Upper East Side North, Total Income: 133153140.21
Zone Name: Penn Station/Madison Sq West, Total Income: 126387228.20
Zone Name: Midtown East, Total Income: 124328670.91
Zone Name: Lincoln Square East, Total Income: 106566263.40
Zone Name: Midtown North, Total Income: 106061840.22


Es natural esperar que los viajes más costosos se encuentren alrededor de los aeropuertos en casi toda ciudad, y NYC no es la excepción. Los aeropuertos se suelen encontrar lejos de los sectores más poblados de una ciudad, lo que significa mayor costo del viaje, además de que muchas veces ya tienen tarifas preestablecidas y no son nada baratas, como es el caso de los taxis de NYC. Así mismo, la ganancia total parece aumentar conforme se analice un sector más privilegiado de NYC.

### 13. Top 10 zonas por tip% (pickup) con mínimo N viajes (N documentado)

In [56]:
N = 100000  # Definimos N como 10,000 viajes para significancia estadística

query_13 = f"""
SELECT z.zone_name, AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct, COUNT(*) as total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY z.zone_name
HAVING COUNT(*) >= {N}
ORDER BY avg_tip_pct DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_13)
        results = cursor.fetchall()
        print(f"Top 10 pickup zones (with at least {N} trips) with the highest average tip percentage in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Average Tip Percentage: {row[1]:.2f}%, Total Trips: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones (with at least 100000 trips) with the highest average tip percentage in the analytics_gold.fct_trips table:
Zone Name: Outside of NYC, Average Tip Percentage: 565.31%, Total Trips: 143788
Zone Name: Jackson Heights, Average Tip Percentage: 25.99%, Total Trips: 132335
Zone Name: Upper East Side South, Average Tip Percentage: 22.30%, Total Trips: 7419847
Zone Name: LaGuardia Airport, Average Tip Percentage: 22.30%, Total Trips: 4796247
Zone Name: Lincoln Square East, Average Tip Percentage: 22.07%, Total Trips: 5069605
Zone Name: Flatiron, Average Tip Percentage: 21.99%, Total Trips: 2585986
Zone Name: Upper East Side North, Average Tip Percentage: 21.76%, Total Trips: 6644657
Zone Name: Upper West Side South, Average Tip Percentage: 21.67%, Total Trips: 4402795
Zone Name: Financial District South, Average Tip Percentage: 21.49%, Total Trips: 491740
Zone Name: Midtown East, Average Tip Percentage: 21.41%, Total Trips: 53799

Es una sorpresa que se observe como punto de recogida Afuera de NYC, sin embargo, al documentación comenta que esto es posible bajo ciertas condiciones: el taxista puede negociar una tarifa distinta a las estándar dependiendo de la situación, y además que se marque como Outside no significa que sea estrictamente hablando fuera de NYC, sino que puede ser una región que no caiga en ninguno de los municipios documentados. Ahora bien, bajo estas condiciones, no es de sorprender que con estas 'tarifas especiales' el porcentaje de propina se dispare, sin embargo, para el resto de zonas es más normal, entendiendo que se trata de zonas menos transitadas y lejanas a las más conocidas.

### 14. Comparación *cash vs card*: viajes, ingreso total, tip%

In [35]:
query_14 = """
SELECT p.payment_type_name AS payment_method, 
       COUNT(*) AS total_viajes, 
       SUM(total_amount) AS ingreso_total,
       AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_payment_type p ON f.payment_type_key = p.payment_type_key
WHERE p.payment_type_key IN (1, 2)  -- 1 = Cash, 2 = Card
GROUP BY p.payment_type_name;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_14)
        results = cursor.fetchall()
        print("Total trips, total income, and average tip percentage by payment method (cash v. card) in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Payment Method: {row[0]}, Total Trips: {row[1]}, Total Income: {row[2]}, Average Tip Percentage: {row[3]:.2f}%")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total trips, total income, and average tip percentage by payment method (cash v. card) in the analytics_gold.fct_trips table:
Payment Method: Cash, Total Trips: 24291078, Total Income: 554895557.53, Average Tip Percentage: 0.00%
Payment Method: Credit card, Total Trips: 120611123, Total Income: 3378589885.38, Average Tip Percentage: 27.14%


La documentación de TLC respecto a los taxis de NYC explica que cuando se paga con efectivo un viaje, la propina que esté incluída no se registra dentro de los datos, con lo que el 0% no es representativo de la realidad, y es imposible saber el valor real con los datos actuales. Ahora bien, dada la cultura de dar propinas en EEUU, no es de sorprender el alto porcentaje que se tiene de propinas, especialmente dado que las personas de clase más alta son las que especialmente pagan con tarjeta de crédito.

### 15. Duración promedio (min) por mes

In [37]:
query_15 = """
SELECT d.month, AVG(EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS avg_duration_min
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_15)
        results = cursor.fetchall()
        print("Average trip duration in minutes per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Average Duration: {row[1]:.2f} minutes")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average trip duration in minutes per month in the analytics_gold.fct_trips table:
Month: January, Average Duration: 15.27 minutes
Month: February, Average Duration: 15.85 minutes
Month: March, Average Duration: 16.53 minutes
Month: April, Average Duration: 17.12 minutes
Month: May, Average Duration: 18.12 minutes
Month: June, Average Duration: 17.76 minutes
Month: July, Average Duration: 17.14 minutes
Month: August, Average Duration: 17.28 minutes
Month: September, Average Duration: 18.74 minutes
Month: October, Average Duration: 18.39 minutes
Month: November, Average Duration: 18.19 minutes
Month: December, Average Duration: 18.48 minutes


Se puede ver una mayor tendencia a alargar los viajes en taxi conforme avanzan los meses de año, teniendo una mayor duración promedio en Diciembre y en Octubre. Esto puede ser debido a las festividades que ocurren en dichos meses, lo que presenta anomalías en los viajes de un ciudadano promedio de NYC, ya que puede visitar familia o irse a una reunión/fiesta específica sin importar la distancia. Además, una mayor movimiento poblacional también causa atrasos en los viajes.

### 16. Distancia promedio por mes

In [38]:
query_16 = """
SELECT d.month, AVG(trip_distance) AS avg_distance
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_16)
        results = cursor.fetchall()
        print("Average trip distance per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Average Distance: {row[1]:.2f} miles")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average trip distance per month in the analytics_gold.fct_trips table:
Month: January, Average Distance: 5.30 miles
Month: February, Average Distance: 5.62 miles
Month: March, Average Distance: 5.71 miles
Month: April, Average Distance: 6.14 miles
Month: May, Average Distance: 6.85 miles
Month: June, Average Distance: 6.51 miles
Month: July, Average Distance: 6.32 miles
Month: August, Average Distance: 6.17 miles
Month: September, Average Distance: 6.75 miles
Month: October, Average Distance: 6.06 miles
Month: November, Average Distance: 6.02 miles
Month: December, Average Distance: 4.94 miles


Comparando estos resultados con los anteriores de tiempo, se puede podría concluir que la causa del aumento de tiempos va más relacionada con el movimiento poblacional que con las festividades de temporada, en tanto la distancia promedio de los viajes no parece aumentar de la misma forma en que los tiempos lo hacen. Así, se tiene que los meses donde se viaja una mayor distancia em promedio son Mayo, Junio y Septiembre. En el caso de Mayo, estos datos van acorde con festividades como el Día de las Madres, dónde los ciudadanos viajan más y mayores distancias, sin importar la rutina diaria como tal.

### 17. Velocidad promedio (mph) por *borough* y franja horaria

In [41]:
query_17 = """
SELECT z.borough, 
       CASE 
           WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 6 AND 9 THEN 'Morning Rush'
           WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 16 AND 19 THEN 'Evening Rush'
           ELSE 'Other'
       END AS time_slot,
       AVG(trip_distance / (NULLIF(EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/3600, 0))) AS avg_speed_mph
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE z.borough IS NOT NULL
GROUP BY z.borough, time_slot
ORDER BY z.borough, avg_speed_mph DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_17)
        results = cursor.fetchall()
        print("Average trip speed in mph by borough and time slot in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Borough: {row[0]}, Time Slot: {row[1]}, Average Speed: {row[2]:.2f} mph")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average trip speed in mph by borough and time slot in the analytics_gold.fct_trips table:
Borough: Bronx, Time Slot: Morning Rush, Average Speed: 218.17 mph
Borough: Bronx, Time Slot: Other, Average Speed: 175.34 mph
Borough: Bronx, Time Slot: Evening Rush, Average Speed: 100.38 mph
Borough: Brooklyn, Time Slot: Morning Rush, Average Speed: 167.38 mph
Borough: Brooklyn, Time Slot: Other, Average Speed: 82.77 mph
Borough: Brooklyn, Time Slot: Evening Rush, Average Speed: 80.84 mph
Borough: EWR, Time Slot: Morning Rush, Average Speed: 101.24 mph
Borough: EWR, Time Slot: Evening Rush, Average Speed: 65.97 mph
Borough: EWR, Time Slot: Other, Average Speed: 61.63 mph
Borough: Manhattan, Time Slot: Morning Rush, Average Speed: 31.20 mph
Borough: Manhattan, Time Slot: Other, Average Speed: 18.69 mph
Borough: Manhattan, Time Slot: Evening Rush, Average Speed: 15.25 mph
Borough: Queens, Time Slot: Morning Rush, Average Speed: 58.32 mph
Borough: Queen

NYC es una ciudad naturalmente muy atareada y ocupada, y esto se vuelve especialmente notorio en las horas pico de cada uno de los días. Se puede ver que durante el 'Morning Rush', entre las 6 y 10 am, se registran velocidades promedio mucho más altas que en el resto de horas, en todos los municipios sin excepción. Esto es consistente tanto con la cantidad de tráfico en tanto en las mañanas el tráfico es menor, como con la necesidad de los ciudadanos de llegar a algún lugar, como sus trabajos, con mayor apuro.

### 18. Percentiles 50 y 90 de duración por borough

In [42]:
query_18 = """
SELECT z.borough, 
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS p50_duration,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS p90_duration
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE z.borough IS NOT NULL
GROUP BY z.borough;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_18)
        results = cursor.fetchall()
        print("50th and 90th percentile of trip duration in minutes by borough in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Borough: {row[0]}, P50 Duration: {row[1]:.2f} minutes, P90 Duration: {row[2]:.2f} minutes")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
50th and 90th percentile of trip duration in minutes by borough in the analytics_gold.fct_trips table:
Borough: Bronx, P50 Duration: 23.13 minutes, P90 Duration: 61.47 minutes
Borough: Brooklyn, P50 Duration: 21.50 minutes, P90 Duration: 48.53 minutes
Borough: EWR, P50 Duration: 0.15 minutes, P90 Duration: 1.32 minutes
Borough: Manhattan, P50 Duration: 11.88 minutes, P90 Duration: 26.30 minutes
Borough: Queens, P50 Duration: 31.88 minutes, P90 Duration: 60.75 minutes
Borough: Staten Island, P50 Duration: 29.03 minutes, P90 Duration: 72.59 minutes
Borough: Unknown, P50 Duration: 14.00 minutes, P90 Duration: 38.18 minutes


Las personas que viven más lejos de los sectores más ocupados de NYC tienden a realizar viajes más largos, y esto se puede ver con los tiempos de viaje desde el Bronx o desde Queens. Sin embargo, también es importante ver el caso de EWR, donde, a pesar de que existen viajes largos, el volumen de viajes muy cortos es tan alto, que no se ven afectados los percentiles.

### 19. Top 10 zonas (pickup) por percentil 90 de duración

In [43]:
query_19 = """
SELECT z.zone_name, 
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS p90_duration
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY z.zone_name
ORDER BY p90_duration DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_19)
        results = cursor.fetchall()
        print("Top 10 pickup zones with the longest 90th percentile trip duration in minutes in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, P90 Duration: {row[1]:.2f} minutes")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones with the longest 90th percentile trip duration in minutes in the analytics_gold.fct_trips table:
Zone Name: Arden Heights, P90 Duration: 124.04 minutes
Zone Name: Freshkills Park, P90 Duration: 102.09 minutes
Zone Name: Far Rockaway, P90 Duration: 99.98 minutes
Zone Name: Hammels/Arverne, P90 Duration: 97.36 minutes
Zone Name: Coney Island, P90 Duration: 92.65 minutes
Zone Name: Eltingville/Annadale/Prince's Bay, P90 Duration: 90.57 minutes
Zone Name: Rockaway Park, P90 Duration: 88.18 minutes
Zone Name: Brighton Beach, P90 Duration: 84.07 minutes
Zone Name: Gravesend, P90 Duration: 83.14 minutes
Zone Name: Cambria Heights, P90 Duration: 80.80 minutes


La situación con Argen Heights es un poco peculiar, ya que el percentil 90 notablemente más alto que el resto de la lista de zonas. Esta situación podría entenderse como que existe un gran número de personas que viajan mucho más tiempo o que en general el tiempo promedio es muy alto para todos. Observando el mapa de NYC, se podría intuir que Arden Heights se encuentra muy lejos del centro de New York, más cerca de la frontera con New Jersey, pero no lo suficientemente lejos como para que las personas busquen alternativas de trabajo en NJ.

### 20. Top 10 rutas *borough to borough* por número de viajes

In [46]:
query_20 = """
SELECT z_pu.borough AS pickup_borough, 
       z_do.borough AS dropoff_borough, 
       COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z_pu ON f.pu_zone_key = z_pu.zone_key
JOIN analytics_gold.dim_zone z_do ON f.do_zone_key = z_do.zone_key
WHERE z_pu.borough IS NOT NULL AND z_do.borough IS NOT NULL
GROUP BY pickup_borough, dropoff_borough
ORDER BY total_viajes DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_20)
        results = cursor.fetchall()
        print("Top 10 pickup and dropoff borough pairs with the most trips in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Pickup Borough: {row[0]}, Dropoff Borough: {row[1]}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup and dropoff borough pairs with the most trips in the analytics_gold.fct_trips table:
Pickup Borough: Manhattan, Dropoff Borough: Manhattan, Trip Count: 133948087
Pickup Borough: Queens, Dropoff Borough: Manhattan, Trip Count: 8271378
Pickup Borough: Manhattan, Dropoff Borough: Queens, Trip Count: 4706915
Pickup Borough: Queens, Dropoff Borough: Queens, Trip Count: 4283222
Pickup Borough: Manhattan, Dropoff Borough: Brooklyn, Trip Count: 3368958
Pickup Borough: Queens, Dropoff Borough: Brooklyn, Trip Count: 2265575
Pickup Borough: Brooklyn, Dropoff Borough: Brooklyn, Trip Count: 1754350
Pickup Borough: Brooklyn, Dropoff Borough: Manhattan, Trip Count: 971410
Pickup Borough: Manhattan, Dropoff Borough: Bronx, Trip Count: 638222
Pickup Borough: Unknown, Dropoff Borough: Unknown, Trip Count: 590788


Era de esperar que el municipio más atareado tanto para recorger como para dejar personas sea Manhattan, con el 81% de viajes aproximadamente ocurriendo entre esos 2 municipios. Sin embargo, es curioso observar que para el caso general de la población, hay más viajes en los que participa Manhattan que dentro de un mismo municipio, i.e. hay más viajes de Queens a Manhattan que de Queens a Queens, o más viajes de Manhattan a Brooklyn que de Brooklyn a Brooklyn. Manhattan siendo el municipio más ocupado y con mayor oportunidades de turismo o trabajo, no es sorpresar que este sea el comportamiento.